# Aviation Speech Enhancement — CDAE Training
Run each cell top to bottom. GPU runtime required (Runtime → Change runtime type → T4 GPU).

In [ ]:
# Cell 1 — Mount Google Drive and verify GPU
from google.colab import drive
drive.mount('/content/drive')

import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name:', torch.cuda.get_device_name(0))

In [ ]:
# Cell 2 — Install extra packages (torch/numpy already on Colab)
!pip install -q librosa soundfile pesq pystoi

In [ ]:
# Cell 3 — Upload your data/generated/ zip to Google Drive, then set this path
# 1. Zip locally:  zip -r generated.zip data/generated/
# 2. Upload generated.zip to your Google Drive
# 3. Update the path below to match where you put it

import os, zipfile

DRIVE_ZIP   = '/content/drive/MyDrive/generated.zip'   # <-- update if needed
EXTRACT_DIR = '/content/data'

os.makedirs(EXTRACT_DIR, exist_ok=True)

print('Extracting...')
with zipfile.ZipFile(DRIVE_ZIP, 'r') as z:
    z.extractall(EXTRACT_DIR)

noisy_files = os.listdir(os.path.join(EXTRACT_DIR, 'generated', 'noisy'))
print(f'Noisy files: {len(noisy_files)}')
print(f'Clean files: {len(os.listdir(os.path.join(EXTRACT_DIR, "generated", "clean")))}')

In [ ]:
# Cell 4 — Preprocessing (STFT -> spectrogram .npy files)
import numpy as np
import librosa
import soundfile as sf

SR       = 16000
N_FFT    = 512
HOP_LEN  = 128

NOISY_WAV = '/content/data/generated/noisy'
CLEAN_WAV = '/content/data/generated/clean'
SPEC_NOISY = '/content/data/spectrograms/noisy'
SPEC_CLEAN = '/content/data/spectrograms/clean'
PHASE_DIR  = '/content/data/phases'

for d in [SPEC_NOISY, SPEC_CLEAN, PHASE_DIR]:
    os.makedirs(d, exist_ok=True)

def to_spectrogram(wav):
    stft  = librosa.stft(wav, n_fft=N_FFT, hop_length=HOP_LEN)
    mag   = np.abs(stft).astype(np.float32)
    phase = np.angle(stft).astype(np.float32)
    return mag, phase

def normalize(mag):
    log_mag = np.log1p(mag)
    mn, mx  = log_mag.min(), log_mag.max()
    if mx - mn < 1e-8: return np.zeros_like(log_mag)
    return ((log_mag - mn) / (mx - mn)).astype(np.float32)

files = sorted([f for f in os.listdir(NOISY_WAV) if f.endswith('.wav')])
print(f'Processing {len(files)} pairs...')

for i, fname in enumerate(files):
    stem = os.path.splitext(fname)[0]
    noisy_audio, _ = librosa.load(os.path.join(NOISY_WAV, fname), sr=SR, mono=True)
    clean_audio, _ = librosa.load(os.path.join(CLEAN_WAV, fname), sr=SR, mono=True)
    noisy_mag, noisy_phase = to_spectrogram(noisy_audio)
    clean_mag, _           = to_spectrogram(clean_audio)
    np.save(os.path.join(SPEC_NOISY, stem + '.npy'), normalize(noisy_mag))
    np.save(os.path.join(SPEC_CLEAN, stem + '.npy'), normalize(clean_mag))
    np.save(os.path.join(PHASE_DIR,  stem + '.npy'), noisy_phase)
    if (i + 1) % 500 == 0 or (i + 1) == len(files):
        print(f'  {i+1}/{len(files)}')

print('Preprocessing done.')

In [ ]:
# Cell 5 — Dataset and model definitions
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

class SpectrogramDataset(Dataset):
    def __init__(self, split='train', val_ratio=0.2):
        stems = sorted([os.path.splitext(f)[0] for f in os.listdir(SPEC_NOISY) if f.endswith('.npy')])
        idx   = int(len(stems) * (1 - val_ratio))
        self.stems = stems[:idx] if split == 'train' else stems[idx:]

    def __len__(self): return len(self.stems)

    def __getitem__(self, i):
        stem  = self.stems[i]
        noisy = np.load(os.path.join(SPEC_NOISY, stem + '.npy'))
        clean = np.load(os.path.join(SPEC_CLEAN, stem + '.npy'))
        return torch.from_numpy(noisy).unsqueeze(0), torch.from_numpy(clean).unsqueeze(0)


class CDAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = nn.Sequential(nn.Conv2d(1,   32,  3, padding=1), nn.BatchNorm2d(32),  nn.ReLU())
        self.enc2 = nn.Sequential(nn.Conv2d(32,  64,  3, padding=1), nn.BatchNorm2d(64),  nn.ReLU())
        self.enc3 = nn.Sequential(nn.Conv2d(64,  128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU())
        self.pool = nn.MaxPool2d(2, 2)
        self.bottleneck = nn.Sequential(nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU())
        self.up1  = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec1 = nn.Sequential(nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU())
        self.up2  = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec2 = nn.Sequential(nn.Conv2d(64,  64,  3, padding=1), nn.BatchNorm2d(64),  nn.ReLU())
        self.up3  = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec3 = nn.Sequential(nn.Conv2d(32,  32,  3, padding=1), nn.BatchNorm2d(32),  nn.ReLU())
        self.output_conv = nn.Conv2d(32, 1, 1)

    def forward(self, x):
        oh, ow = x.shape[2], x.shape[3]
        ph = (8 - oh % 8) % 8; pw = (8 - ow % 8) % 8
        if ph or pw: x = F.pad(x, (0, pw, 0, ph))
        e1 = self.enc1(x);      e1p = self.pool(e1)
        e2 = self.enc2(e1p);    e2p = self.pool(e2)
        e3 = self.enc3(e2p);    e3p = self.pool(e3)
        b  = self.bottleneck(e3p)
        d1 = self.dec1(self.up1(b))
        d2 = self.dec2(self.up2(d1))
        d3 = self.dec3(self.up3(d2))
        return torch.sigmoid(self.output_conv(d3))[:, :, :oh, :ow]


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

In [ ]:
# Cell 6 — Training
import time

EPOCHS     = 30
BATCH_SIZE = 32
LR         = 1e-3
MODEL_OUT  = '/content/drive/MyDrive/cdae_best.pth'  # saved directly to Drive

train_ds = SpectrogramDataset('train')
val_ds   = SpectrogramDataset('val')
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f'Train: {len(train_ds)}  Val: {len(val_ds)}')

model     = CDAE().to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

best_val = float('inf')
train_losses, val_losses = [], []

print(f'\n{"Epoch":>6}  {"Train":>10}  {"Val":>10}  {"LR":>8}  {"Time":>7}  Status')
print('-' * 58)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    model.train()
    tl = 0
    for n, c in train_loader:
        n, c = n.to(device), c.to(device)
        optimizer.zero_grad()
        loss = criterion(model(n), c)
        loss.backward()
        optimizer.step()
        tl += loss.item() * n.size(0)
    tl /= len(train_ds)

    model.eval()
    vl = 0
    with torch.no_grad():
        for n, c in val_loader:
            n, c = n.to(device), c.to(device)
            vl += criterion(model(n), c).item() * n.size(0)
    vl /= len(val_ds)

    train_losses.append(tl); val_losses.append(vl)
    scheduler.step(vl)
    lr = optimizer.param_groups[0]['lr']
    status = ''
    if vl < best_val:
        best_val = vl
        torch.save({'epoch': epoch, 'model_state': model.state_dict(),
                    'best_val': best_val, 'train_losses': train_losses, 'val_losses': val_losses}, MODEL_OUT)
        status = 'SAVED'
    print(f'{epoch:>6}  {tl:>10.6f}  {vl:>10.6f}  {lr:>8.2e}  {time.time()-t0:>6.1f}s  {status}')

print(f'\nBest val loss: {best_val:.6f}')
print(f'Model saved to Google Drive: {MODEL_OUT}')

In [ ]:
# Cell 7 — Plot loss curve
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 4))
plt.plot(train_losses, label='Train loss')
plt.plot(val_losses,   label='Val loss', linestyle='--')
plt.xlabel('Epoch'); plt.ylabel('MSE Loss')
plt.title('CDAE Training Loss'); plt.legend(); plt.tight_layout()
plt.savefig('/content/drive/MyDrive/loss_curve.png', dpi=150)
plt.show()
print('Loss curve saved to Drive.')